# Take best runs from every Sweep in wandb to analyze it further

In [1]:
# imports
import sys
import os
import json

# helpers
sys.path.insert(0, "../../")

from wandb_helper import (
    list_sweeps, 
    get_best_runs_all_sweeps, 
    get_best_by_model_type, 
    get_training_stats
)

/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
# data directory and output path
SWEEPS_PATH         = "sweeps.json"
BEST_RUNS_PATH      = "best_runs.json"
MODEL_TYPE_PATH     = "best_by_model_type.json"
TRAINING_STATS_PATH = "training_stats.json"

## 1. List all created sweeps in the project

In [3]:
# list sweeps and cache them
if not os.path.exists(SWEEPS_PATH):
    sweeps = list_sweeps(output_path=SWEEPS_PATH)
    print(f"Saved: {SWEEPS_PATH}")
else:
    with open(SWEEPS_PATH, encoding="utf-8") as f:
        sweeps = json.load(f)
    print(f"Loaded from cache: {SWEEPS_PATH}")

# print sweeps
print(f"\n{'sweep_id':15s} sweep_name\n")
for sweep in sweeps:
    print(f"{sweep['sweep_id']:15s} {sweep['sweep_name']}")

Loaded from cache: sweeps.json

sweep_id        sweep_name

s4wv3fsz        PlainNeuralNetwork_all_categories
fcol9nkq        XGBoost_top_9_features
fz1dt1vk        RandomForest_top_9_features
of6bkya6        XGBoost_top_35_features
qhni79i9        RandomForest_top_35_features
8g4miv1l        RandomForest_general_material_ray_categories
n3yw24wq        XGBoost_general_material_ray_categories
nh6ngran        XGBoost_all_categories
o510n6kh        RandomForest_all_categories


## 2. Fetch the best 10 runs per sweep

In [4]:
# get best runs for all sweeps and cache them
if not os.path.exists(BEST_RUNS_PATH):
    results = get_best_runs_all_sweeps(
        top_n=10,
        output_path=BEST_RUNS_PATH,
        primary_metric="val/f1_macro",
        secondary_metric="val/mcc",
    )
    print(f"Saved: {BEST_RUNS_PATH}")
else:
    with open(BEST_RUNS_PATH, encoding="utf-8") as f:
        results = json.load(f)
    print(f"Loaded from cache: {BEST_RUNS_PATH}")

Loaded from cache: best_runs.json


In [5]:
# check if we have results for all sweeps
for sweep_id, runs in results["runs_per_sweep"].items():
    if not runs:
        continue
    sweep_name = runs[0]["sweep_name"]

    print(f"\nSweep: {sweep_name}  ({sweep_id})\n")
    print(f"{'Nr.':4}{'model_type':27s}{'over':7s}{'f1':8}{'mcc':8}run_name\n")

    for idx, run in enumerate(runs, 1):
        f1 = run["metrics"].get("val/f1_macro", float("nan"))
        mcc = run["metrics"].get("val/mcc", float("nan"))
        print(f"{idx:<4}{str(run['model_type']):27s}{str(run['oversampling']):7s}{f1:.4f}  {mcc:.4f}  {run['run_name']}")


Sweep: PlainNeuralNetwork_all_categories  (s4wv3fsz)

Nr. model_type                 over   f1      mcc     run_name

1   plain_neural_network       True   0.6784  0.6468  n_hidden_512_dropout_0.49636938032493494_lr_3.3912373312779774e-05_batch_size_128_epochs_200_training_oversample_True_82d138
2   plain_neural_network       True   0.6772  0.6581  n_hidden_512_dropout_0.39701613308724826_lr_6.0725787145125745e-05_batch_size_256_epochs_200_training_oversample_True_dad4a4
3   plain_neural_network       True   0.6770  0.6700  n_hidden_128_dropout_0.1674813663340506_lr_0.00035284113156915797_batch_size_128_epochs_200_training_oversample_True_93d48b
4   plain_neural_network       True   0.6768  0.6583  n_hidden_128_dropout_0.16356546820388257_lr_0.000241011442037694_batch_size_128_epochs_200_training_oversample_True_fb6aa7
5   plain_neural_network       True   0.6736  0.6538  n_hidden_768_dropout_0.47343383387714455_lr_0.00011945779800036688_batch_size_256_epochs_200_training_oversample_T

## 3. Fetch the best run per model type with and without oversampling

In [6]:
# get best runs by model type and cache them
if not os.path.exists(MODEL_TYPE_PATH):
    best_by_model_type = get_best_by_model_type(
        input_path=BEST_RUNS_PATH,
        output_path=MODEL_TYPE_PATH,
    )
    print(f"Saved: {MODEL_TYPE_PATH}")
else:
    with open(MODEL_TYPE_PATH, encoding="utf-8") as f:
        best_by_model_type = json.load(f)
    print(f"Loaded from cache: {MODEL_TYPE_PATH}")

# inspect best runs by model type
for model_type, groups in best_by_model_type.items():
    print("\n" + model_type)
    for label, bests in groups.items():
        print(f"  {label}:")
        for metric_key, run in bests.items():
            metric = metric_key.replace("best_", "").replace("_", "/", 1)
            val = run["metrics"].get(metric)
            val_str = f"{val:.4f}" if val is not None else "n/a"
            print(f"    {metric_key}: {val_str}")

Loaded from cache: best_by_model_type.json

plain_neural_network
  oversampling:
    best_val_f1_macro: 0.6784
    best_val_mcc: 0.6468
  no_oversampling:
    best_val_f1_macro: 0.6660
    best_val_mcc: 0.6493

xgboost
  oversampling:
    best_val_f1_macro: 0.7276
    best_val_mcc: 0.7106
  no_oversampling:
    best_val_f1_macro: 0.7181
    best_val_mcc: 0.7148

random_forest
  oversampling:
    best_val_f1_macro: 0.7143
    best_val_mcc: 0.6952
  no_oversampling:
    best_val_f1_macro: 0.7044
    best_val_mcc: 0.6776


## 4. Fetch and analyse the training duration statistics between Random Forest and XGBoost

In [7]:
# get training duration statistics and cache them
if not os.path.exists(TRAINING_STATS_PATH):
    stats = get_training_stats(
        models=("random_forest", "xgboost"),
        output_path=TRAINING_STATS_PATH,
    )
    print(f"Saved: {TRAINING_STATS_PATH}\n")
else:
    with open(TRAINING_STATS_PATH, encoding="utf-8") as f:
        stats = json.load(f)
    print(f"Loaded from cache: {TRAINING_STATS_PATH}\n")

# inspect training duration statistics
for model_type, groups in stats.items():
    print(f"  {model_type}")
    for label, stats in groups.items():
        print(f"    {label:15s}  count={stats['count']:4d}  mean ={stats['mean_seconds']:5.1f}s  median ={stats['median_seconds']:5.1f}s")

Loaded from cache: training_stats.json

  random_forest
    no_oversampling  count= 439  mean = 14.9s  median = 14.4s
    oversampling     count=1732  mean = 16.3s  median = 13.8s
    total            count=2171  mean = 16.0s  median = 14.1s
  xgboost
    oversampling     count=2811  mean = 39.4s  median = 38.4s
    no_oversampling  count= 977  mean = 32.5s  median = 31.9s
    total            count=3788  mean = 37.6s  median = 36.4s
